# Segment 3 — Conversation Memory Patterns
### Buffer · Summary · Sliding Window — runnable examples

This notebook is the hands-on companion to Segment 3. It builds all three memory patterns from scratch, then runs the same 10-turn conversation through all three so you can watch them behave differently.

**No vector database. No new infrastructure.** Just history, some bookkeeping, and one optional LLM call (for Summary Memory).

Run the cells top to bottom. Estimated time: 10 minutes.

## Setup

We need one thing: a way to call an LLM to do the actual summarizing in Part 2.

- **If you have an OpenAI API key**, paste it in the cell below (or add it as a Colab secret named `OPENAI_API_KEY`) and everything runs against the real OpenAI API.
- **If you don't have a key handy**, leave it blank — the notebook falls back to a simple offline "mock" so every cell still runs and you can see the shape of the pattern. It's clearly labeled below.

In [1]:
# Install the OpenAI SDK (only needed if you're using a real API key)
!pip install openai --quiet

In [2]:
import os
import re

# --- OPTION A: Colab secrets (recommended) ---
# Click the key icon in the left sidebar, add a secret named OPENAI_API_KEY, then run this cell.
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = None

# --- OPTION B: paste your key directly (only for local/throwaway testing) ---
# OPENAI_API_KEY = "sk-..."

USE_REAL_LLM = bool(OPENAI_API_KEY)

if USE_REAL_LLM:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    def llm_complete(prompt, model="gpt-4o-mini", max_tokens=300):
        resp = client.chat.completions.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.choices[0].message.content
    print("Using the real OpenAI API - live LLM calls are on.")

else:
    # OFFLINE STUB MODE - no API key found.
    # This still needs to do two DIFFERENT jobs honestly, or the demo lies to you:
    #   1. Summarizing history down (used by Summary Memory)
    #   2. Answering a question from a given context (used by the exercise)
    # A real LLM does both well. The stub below is deliberately simple so you
    # can SEE the shape of each pattern - it is not a real language model.

    def _stub_summarize(prompt):
        # Naive + deliberately lossy: keeps only the most RECENT few user
        # turns. This mirrors a real risk with summarization - a model with
        # a lazy prompt will happily favor recent content and drop early
        # details (like a name mentioned back at turn 2). Try tightening the
        # summarization prompt in maybe_summarize() to fix this for real.
        lines = [l for l in prompt.split("\n") if l.strip().startswith("user:")]
        kept = [l.split("user:", 1)[1].strip().split(".")[0] for l in lines[-3:]]
        return "Summary (offline stub): " + " | ".join(kept)

    def _stub_answer(prompt):
        # Very simple keyword lookup - looks for a 'my name is X' style
        # pattern anywhere in the given context and answers from it. If it's
        # not in the context you handed it, it honestly says so, instead of
        # making something up.
        context = prompt.split("Now answer this question")[0]
        match = re.search(r"name is (\w+)", context, re.IGNORECASE)
        if match:
            return f"(offline stub) Your name is {match.group(1)}."
        return "(offline stub) I don't have that information in what you've told me so far."

    def llm_complete(prompt, model=None, max_tokens=None):
        if prompt.strip().startswith("Summarize this conversation"):
            return _stub_summarize(prompt)
        return _stub_answer(prompt)

    print("No API key found - running in OFFLINE STUB mode.")
    print("   Summaries are a crude, deliberately lossy keyword join - not a real summary.")
    print("   Question-answering is a simple keyword lookup - not a real model.")
    print("   Add OPENAI_API_KEY above (see Option A/B) for live, real results.")


Using the real OpenAI API - live LLM calls are on.


---
## Part 1 — Buffer Memory: The Shoebox

Keep everything. Every turn goes back to the model, every time. Simple, but it grows without bound.

In [3]:
def buffer_add(history, user_msg, assistant_reply):
    """Append a turn to the shoebox. Nothing gets thrown away."""
    history.append({"role": "user", "content": user_msg})
    history.append({"role": "assistant", "content": assistant_reply})
    return history


# --- demo ---
history = []
history = buffer_add(history, "Hey, can you help me plan a trip to Japan?", "Of course! When are you thinking of going?")
history = buffer_add(history, "Sometime in April, for the cherry blossoms.", "Great timing — early April is usually peak bloom in Tokyo and Kyoto.")
history = buffer_add(history, "Cool, what cities should I visit?", "Tokyo, Kyoto, and maybe Osaka for food.")

for turn in range(4, 21):
    history = buffer_add(history, f"Follow-up question #{turn}", f"Follow-up answer #{turn}")

print(f"Turns stored: {len(history)//2}")
print(f"Approx size of what gets sent back to the model every turn: {len(str(history))} characters")
print()
print("First turn (still there, unshrunk):")
print(" ", history[0])

Turns stored: 20
Approx size of what gets sent back to the model every turn: 2389 characters

First turn (still there, unshrunk):
  {'role': 'user', 'content': 'Hey, can you help me plan a trip to Japan?'}


Notice: the shoebox never shrinks. By turn 20, you're still carrying turn 1 around, every single time you talk. That's the O(n) growth from the slide — turn 1 is small, turn 20 means resending everything that's ever been said.

---
## Part 2 — Summary Memory: Shrinking It Down

Same idea as buffer memory, until you cross a threshold. Then you ask the model to compress everything so far into one note, and swap it in.

In [4]:
def format_turns(history):
    """Turn the history list into plain text for the LLM to summarize."""
    lines = []
    for msg in history:
        lines.append(f"{msg['role']}: {msg['content']}")
    return "\n".join(lines)


def maybe_summarize(history, n=10):
    """If the shoebox is under the limit, do nothing.
    Once it crosses the line, shrink it down to one summary message."""
    if len(history) < n:
        return history

    text = format_turns(history)
    summary = llm_complete(f"Summarize this conversation history in 2-3 sentences, "
                            f"keeping any names, preferences, or decisions the user "
                            f"mentioned:\n\n{text}")
    return [{"role": "system", "content": summary}]


# --- demo: reuse the 20-turn buffer history from Part 1 ---
print(f"Before: {len(history)} messages in history")

shrunk = maybe_summarize(history, n=10)

print(f"After:  {len(shrunk)} message(s) in history")
print()
print("The summary:")
print(" ", shrunk[0]["content"])

Before: 40 messages in history
After:  1 message(s) in history

The summary:
  The user is planning a trip to Japan in April to enjoy the cherry blossoms and is interested in visiting Tokyo, Kyoto, and possibly Osaka for its food. The conversation includes numerous follow-up questions, but the specifics of those questions were not provided.


Notice: the trade. Cost stays bounded no matter how long the conversation runs — but summarizing cost you an extra call to the model, and the model chose what to keep. You don't get a say in the moment.

---
## Part 3 — Sliding Window Memory: The Rearview Mirror

No LLM call, no shrinking logic. Just keep the last N turn-pairs and drop everything older. Pure bookkeeping.

In [5]:
def sliding_window(history, n=4):
    """Keep last N turn-pairs"""
    # Each pair = 1 user + 1 assistant
    pairs = []
    for i in range(0, len(history), 2):
        pairs.append(history[i:i+2])
    recent = pairs[-n:]
    return [t for p in recent for t in p]


# --- demo: reuse the 20-turn buffer history from Part 1 ---
windowed = sliding_window(history, n=4)

print(f"Full history:    {len(history)//2} turns")
print(f"After windowing: {len(windowed)//2} turns (only the last 4)")
print()
print("What's actually left in the window:")
for msg in windowed:
    print(" ", msg["role"], "->", msg["content"][:60])

Full history:    20 turns
After windowing: 4 turns (only the last 4)

What's actually left in the window:
  user -> Follow-up question #17
  assistant -> Follow-up answer #17
  user -> Follow-up question #18
  assistant -> Follow-up answer #18
  user -> Follow-up question #19
  assistant -> Follow-up answer #19
  user -> Follow-up question #20
  assistant -> Follow-up answer #20


Notice: it costs nothing extra, but anything older than the window is just gone — not shrunk into a summary, gone. That's the trade: free and instant, blind to anything outside the mirror.

---
## Part 4 — The Exercise: Same Conversation, Three Agents

Now the real test. One 10-turn conversation. Turn 2, the user mentions their name is **Aria**. Then eight turns of unrelated small talk. Turn 10, they ask: *"What's my name?"*

Does each pattern still know, by turn 10?

In [6]:
def build_test_conversation():
    """10 turns. The name 'Aria' is buried in turn 2."""
    convo = []
    convo += [{"role": "user", "content": "Hi there!"},
              {"role": "assistant", "content": "Hello! How can I help you today?"}]                      # turn 1
    convo += [{"role": "user", "content": "My name is Aria, by the way."},
              {"role": "assistant", "content": "Nice to meet you, Aria!"}]                                # turn 2  <-- the detail
    convo += [{"role": "user", "content": "What's a good weeknight dinner idea?"},
              {"role": "assistant", "content": "A sheet-pan chicken and veggie bake is quick and easy."}] # turn 3
    convo += [{"role": "user", "content": "Any good movies out right now?"},
              {"role": "assistant", "content": "Depends on the genre — action, comedy, or drama?"}]       # turn 4
    convo += [{"role": "user", "content": "I like sci-fi."},
              {"role": "assistant", "content": "Then you might enjoy a good near-future thriller."}]      # turn 5
    convo += [{"role": "user", "content": "What's the weather like in Seattle in October?"},
              {"role": "assistant", "content": "Cool and rainy, average highs around 60°F."}]             # turn 6
    convo += [{"role": "user", "content": "Can you recommend a good book?"},
              {"role": "assistant", "content": "If you like sci-fi, try Project Hail Mary."}]             # turn 7
    convo += [{"role": "user", "content": "How do I make cold brew coffee?"},
              {"role": "assistant", "content": "Steep coarse grounds in cold water for 12-18 hours."}]    # turn 8
    convo += [{"role": "user", "content": "What's a good beginner workout routine?"},
              {"role": "assistant", "content": "A simple full-body routine 3x a week is a great start."}] # turn 9
    convo += [{"role": "user", "content": "What's my name?"},
              {"role": "assistant", "content": "<<< TO BE ANSWERED >>>"}]                                 # turn 10 <-- the test
    return convo


test_convo = build_test_conversation()
question_only = test_convo[:-1]  # everything up to (not including) the final assistant placeholder

print(f"Test conversation built: {len(test_convo)//2} turns")
print("Turn 2 (the detail):", test_convo[2]["content"])
print("Turn 10 (the test)  :", test_convo[-2]["content"])

Test conversation built: 10 turns
Turn 2 (the detail): My name is Aria, by the way.
Turn 10 (the test)  : What's my name?


In [7]:
def ask_with_context(context, question="What's my name?"):
    """Feed a context + the test question to the LLM and get the answer."""
    text = format_turns(context)
    prompt = (f"Here is the conversation so far:\n\n{text}\n\n"
              f"Now answer this question based only on the context above: {question}")
    return llm_complete(prompt, max_tokens=60)


# --- Pattern 1: Buffer Memory (full history, nothing dropped) ---
buffer_context = question_only  # the whole thing
buffer_answer = ask_with_context(buffer_context)

# --- Pattern 2: Sliding Window (last 4 turn-pairs only) ---
window_context = sliding_window(question_only, n=4)
window_answer = ask_with_context(window_context)

# --- Pattern 3: Summary Memory (shrunk down after turn 9) ---
# Summarize everything BEFORE the final question, then ask on top of the summary
summary_context = maybe_summarize(question_only[:-2], n=1)  # force summarization for the demo
summary_context = summary_context + question_only[-2:]      # add the actual question back on top
summary_answer = ask_with_context(summary_context)

print("BUFFER MEMORY   ->", buffer_answer)
print()
print("SLIDING WINDOW  ->", window_answer)
print()
print("SUMMARY MEMORY  ->", summary_answer)

BUFFER MEMORY   -> Your name is Aria.

SLIDING WINDOW  -> I'm sorry, but you haven't mentioned your name in the conversation.

SUMMARY MEMORY  -> Your name is Aria.


### What to expect

- **Buffer memory** — should get it right. Nothing was ever dropped; turn 2 is still sitting right there in the full history.
- **Sliding window (N=4)** — turn 2 is 8 turns back from turn 10, well outside a window of 4. It very likely won't know the name. That's not a bug — that's the trade-off.
- **Summary memory** — depends entirely on whether the model judged "the user's name is Aria" important enough to keep during compression. With a real LLM and a good summarization prompt, it usually survives. With the offline stub, check the printed summary above to see if "Aria" made the cut.

Same conversation. Three different agents. Three different answers to the exact same question — that's the whole segment in one exercise.

---
## Bonus — Try it yourself

Some things to experiment with:

1. **Change the window size.** Try `sliding_window(question_only, n=6)` — does it get far enough back to catch turn 2?
2. **Move the detail.** Put "My name is Aria" at turn 8 instead of turn 2, and rerun the sliding window — now it should be caught.
3. **Tighten the summarization prompt.** Edit the prompt inside `maybe_summarize` to explicitly say *"Always preserve the user's name if mentioned"* — see if that makes summary memory more reliable.
4. **Combine patterns.** Try feeding the sliding window's output *into* `maybe_summarize` instead of using either alone — that's the production pattern mentioned in the Choosing the Right Pattern section: recent turns in the window, older turns folded into a running summary instead of dropped entirely.

In [9]:
# -*- coding: utf-8 -*-
"""
O'Reilly Live Training — AI Agent Memory Essentials
Segment 3: Conversation Memory Patterns
🎯 Bonus — Try it yourself (all 4 experiments)


"""

# ============================================================
# Bonus 1 — Change the window size: does n=6 reach back far enough?
# ============================================================

print("=" * 60)
print("BONUS 1 — Widening the sliding window")
print("=" * 60)

for n in [4, 6, 8]:
    ctx = sliding_window(question_only, n=n)
    answer = ask_with_context(ctx)
    turns_covered = len(ctx) // 2
    print(f"\nWindow size n={n} (covers last {turns_covered} turns):")
    print(" ", answer)

print("\n💡 Turn 2 is 8 turns back from turn 10. A window of n=6 only reaches")
print("   back to turn 5 — still misses it. You need n=8 (or larger) to pull")
print("   turn 2 back into view. This is the exact 'how big should my window")
print("   be' trade-off from the slide: bigger window = more coverage,")
print("   but back to paying the buffer-memory token cost.")




BONUS 1 — Widening the sliding window

Window size n=4 (covers last 3 turns):
  I'm sorry, but you haven't shared your name in our conversation.

Window size n=6 (covers last 5 turns):
  I'm sorry, but your name hasn't been mentioned in our conversation.

Window size n=8 (covers last 7 turns):
  I'm sorry, but I don't have your name based on our previous conversation.

💡 Turn 2 is 8 turns back from turn 10. A window of n=6 only reaches
   back to turn 5 — still misses it. You need n=8 (or larger) to pull
   turn 2 back into view. This is the exact 'how big should my window
   be' trade-off from the slide: bigger window = more coverage,
   but back to paying the buffer-memory token cost.


In [10]:
# ============================================================
# Bonus 2 — Move the detail to turn 8, rerun sliding window (n=4)
# ============================================================

print("\n\n" + "=" * 60)
print("BONUS 2 — Moving 'My name is Aria' from turn 2 to turn 8")
print("=" * 60)

def build_test_conversation_moved():
    """Same 10-turn conversation, but the name now lands at turn 8
    instead of turn 2. Turn 2 becomes ordinary small talk."""
    convo = []
    convo += [{"role": "user", "content": "Hi there!"},
              {"role": "assistant", "content": "Hello! How can I help you today?"}]                      # turn 1
    convo += [{"role": "user", "content": "What's a good weeknight dinner idea?"},
              {"role": "assistant", "content": "A sheet-pan chicken and veggie bake is quick and easy."}] # turn 2 (was turn 3)
    convo += [{"role": "user", "content": "Any good movies out right now?"},
              {"role": "assistant", "content": "Depends on the genre — action, comedy, or drama?"}]       # turn 3 (was turn 4)
    convo += [{"role": "user", "content": "I like sci-fi."},
              {"role": "assistant", "content": "Then you might enjoy a good near-future thriller."}]      # turn 4 (was turn 5)
    convo += [{"role": "user", "content": "What's the weather like in Seattle in October?"},
              {"role": "assistant", "content": "Cool and rainy, average highs around 60°F."}]             # turn 5 (was turn 6)
    convo += [{"role": "user", "content": "Can you recommend a good book?"},
              {"role": "assistant", "content": "If you like sci-fi, try Project Hail Mary."}]             # turn 6 (was turn 7)
    convo += [{"role": "user", "content": "How do I make cold brew coffee?"},
              {"role": "assistant", "content": "Steep coarse grounds in cold water for 12-18 hours."}]    # turn 7 (was turn 8)
    convo += [{"role": "user", "content": "My name is Aria, by the way."},
              {"role": "assistant", "content": "Nice to meet you, Aria!"}]                                # turn 8  <-- moved here
    convo += [{"role": "user", "content": "What's a good beginner workout routine?"},
              {"role": "assistant", "content": "A simple full-body routine 3x a week is a great start."}] # turn 9
    convo += [{"role": "user", "content": "What's my name?"},
              {"role": "assistant", "content": "<<< TO BE ANSWERED >>>"}]                                 # turn 10 <-- the test
    return convo


moved_convo = build_test_conversation_moved()
moved_question_only = moved_convo[:-1]

moved_window = sliding_window(moved_question_only, n=4)
moved_answer = ask_with_context(moved_window)

print(f"\nTurn 8 (the detail, now inside a window of 4): {moved_convo[14]['content']}")
print(f"\nSLIDING WINDOW (n=4) with name at turn 8 ->")
print(" ", moved_answer)

print("\n💡 With the same n=4 window that failed before, the name now sits")
print("   only 2 turns back from the question — comfortably inside the")
print("   window — so it should be caught. Nothing about the pattern")
print("   changed; only where the detail happened to land.")




BONUS 2 — Moving 'My name is Aria' from turn 2 to turn 8

Turn 8 (the detail, now inside a window of 4): My name is Aria, by the way.

SLIDING WINDOW (n=4) with name at turn 8 ->
  Your name is Aria.

💡 With the same n=4 window that failed before, the name now sits
   only 2 turns back from the question — comfortably inside the
   window — so it should be caught. Nothing about the pattern
   changed; only where the detail happened to land.


In [11]:
# ============================================================
# Bonus 3 — Tighten the summarization prompt to protect names
# ============================================================

print("\n\n" + "=" * 60)
print("BONUS 3 — A stricter summarization prompt")
print("=" * 60)

def maybe_summarize_strict(history, n=10):
    """Same as maybe_summarize, but the prompt explicitly instructs
    the model to never drop the user's name."""
    if len(history) < n:
        return history

    text = format_turns(history)
    summary = llm_complete(
        "Summarize this conversation history in 2-3 sentences. "
        "Always preserve the user's name if mentioned, along with any "
        "other preferences or decisions the user stated, even if they "
        "were mentioned only once early in the conversation:\n\n" + text
    )
    return [{"role": "system", "content": summary}]


# Original (loose) prompt, for comparison
loose_summary = maybe_summarize(question_only[:-2], n=1)
loose_context = loose_summary + question_only[-2:]
loose_answer = ask_with_context(loose_context)

# Strict (name-protecting) prompt
strict_summary = maybe_summarize_strict(question_only[:-2], n=1)
strict_context = strict_summary + question_only[-2:]
strict_answer = ask_with_context(strict_context)

print("\nOriginal summary:")
print(" ", loose_summary[0]["content"])
print("Answer:", loose_answer)

print("\nStrict (name-protecting) summary:")
print(" ", strict_summary[0]["content"])
print("Answer:", strict_answer)

print("\n💡 With a real LLM, an explicit 'always preserve the name' instruction")
print("   in the summarization prompt should make Aria show up in the")
print("   summary far more reliably. The offline stub in this notebook is a")
print("   naive keyword-join, though — it isn't following instructions at all,")
print("   so both stub outputs will look identical until you plug in a real key.")



BONUS 3 — A stricter summarization prompt

Original summary:
  Aria is looking for weeknight dinner ideas, and a sheet-pan chicken and veggie bake was suggested. She enjoys sci-fi movies and books, with a recommendation for *Project Hail Mary*, and is interested in learning how to make cold brew coffee as well as beginner workout routines. Additionally, she inquired about the weather in Seattle in October, expecting cool and rainy conditions.
Answer: Your name is Aria.

Strict (name-protecting) summary:
  Aria is interested in weeknight dinner ideas, movie recommendations (particularly in the sci-fi genre), and asked about the weather in Seattle in October. She also seeks advice on making cold brew coffee and beginner workout routines. Additionally, she received a book suggestion, "Project Hail Mary," which aligns with her sci-fi interest.
Answer: Your name is Aria.

💡 With a real LLM, an explicit 'always preserve the name' instruction
   in the summarization prompt should make Aria 

In [12]:

# ============================================================
# Bonus 4 — Combine patterns: window + summary of what falls outside it
# ============================================================

print("\n\n" + "=" * 60)
print("BONUS 4 — Combining sliding window with summary memory")
print("=" * 60)

def windowed_with_summary(history, window_size=4, summarize_fn=maybe_summarize_strict):
    """Production pattern: keep the last `window_size` turn-pairs verbatim,
    and fold everything OLDER than the window into a running summary
    instead of dropping it. Returns [summary_message] + recent_turns.
    """
    pairs = []
    for i in range(0, len(history), 2):
        pairs.append(history[i:i + 2])

    recent_pairs = pairs[-window_size:]
    older_pairs = pairs[:-window_size]

    recent_turns = [t for p in recent_pairs for t in p]

    if not older_pairs:
        return recent_turns

    older_turns = [t for p in older_pairs for t in p]
    # Force summarization regardless of length threshold — we WANT
    # everything outside the window folded in, however short it is.
    summary_msgs = summarize_fn(older_turns, n=1)

    return summary_msgs + recent_turns


combined_context = windowed_with_summary(question_only, window_size=4)
combined_answer = ask_with_context(combined_context)

print(f"\nCombined context: {len(combined_context)} message(s) total")
print("  - 1 summary message covering turns 1-6 (outside the window)")
print("  - plus the last 4 turn-pairs (turns 7-10) kept verbatim")
print("\nWhat the summary folded in:")
print(" ", combined_context[0]["content"])
print("\nCOMBINED (window + summary) ->")
print(" ", combined_answer)

print("\n💡 This is the best of both: recent context stays word-for-word")
print("   accurate (no summarization drift on what was JUST said), while")
print("   older turns aren't silently discarded like plain sliding window —")
print("   they're compressed instead. Token cost stays roughly bounded no")
print("   matter how long the conversation runs, and Aria survives because")
print("   she's inside the summarized portion, not the dropped portion.")


# ============================================================
# Recap
# ============================================================
print("\n\n" + "=" * 60)
print("🎯 Bonus experiments complete:")
print("  1. n=6 still misses turn 2; need n>=8 to reach back that far")
print("  2. Moving the detail to turn 8 makes the SAME n=4 window catch it")
print("  3. An explicit 'preserve names' instruction makes summarization")
print("     more reliable (most visible with a real LLM, not the stub)")
print("  4. Window + summary combo bounds cost AND avoids silent data loss")
print("=" * 60)



BONUS 4 — Combining sliding window with summary memory

Combined context: 8 message(s) total
  - 1 summary message covering turns 1-6 (outside the window)
  - plus the last 4 turn-pairs (turns 7-10) kept verbatim

What the summary folded in:
  Aria is looking for weeknight dinner ideas and received a suggestion for a sheet-pan chicken and veggie bake. She expressed an interest in sci-fi movies, and the conversation also touched on the cool and rainy weather in Seattle during October, with average highs around 60°F.

COMBINED (window + summary) ->
  Your name is Aria.

💡 This is the best of both: recent context stays word-for-word
   accurate (no summarization drift on what was JUST said), while
   older turns aren't silently discarded like plain sliding window —
   they're compressed instead. Token cost stays roughly bounded no
   matter how long the conversation runs, and Aria survives because
   she's inside the summarized portion, not the dropped portion.


🎯 Bonus experiments com